In [2]:
import requests          # For making HTTP requests to download files from the web
import zipfile           # DWD files come as .zip archives — this extracts them
import io                # Lets us treat downloaded bytes as a file without saving first
import os                # For creating folders, checking file paths
import pandas as pd      # Data manipulation
from tqdm import tqdm    # Progress bar — shows download progress nicely

print("✅ Imports done")

✅ Imports done


In [3]:
# DWD organizes data into two buckets:
# 'historical' = older data (fully quality-checked)
# 'recent'     = last ~1.5 years (preliminary, less checked)
# We need BOTH to cover 2004-2024

STATION_ID = "00433"   # Berlin-Tempelhof — always zero-padded to 5 digits

# Base URL where DWD stores hourly air temperature data
BASE_URL = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/air_temperature/"

HISTORICAL_URL = BASE_URL + "historical/"
RECENT_URL     = BASE_URL + "recent/"

print(f"Station ID     : {STATION_ID}")
print(f"Historical URL : {HISTORICAL_URL}")
print(f"Recent URL     : {RECENT_URL}")

Station ID     : 00433
Historical URL : https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/air_temperature/historical/
Recent URL     : https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/air_temperature/recent/


In [4]:
def get_station_filename(index_url, station_id):
    """
    DWD stores files like:
    stundenwerte_TU_00433_19880101_20231231_akt.zip
    
    We fetch the directory listing and search for our station's file.
    
    Args:
        index_url  : URL of the DWD directory (historical or recent)
        station_id : 5-digit station ID string e.g. "00433"
    
    Returns:
        filename : the exact filename for our station
    """
    
    # Step 1: Fetch the HTML of the directory page
    response = requests.get(index_url)
    
    # Step 2: The page is plain HTML — we scan it line by line
    # looking for a line that contains our station ID
    for line in response.text.split('\n'):
        if f'_{station_id}_' in line and line.endswith('.zip"'):
            # Step 3: Extract just the filename from the HTML anchor tag
            # HTML looks like: <a href="stundenwerte_TU_00433_...zip">
            filename = line.split('href="')[1].split('"')[0]
            return filename
    
    return None  # Return None if not found

# Test it
hist_filename   = get_station_filename(HISTORICAL_URL, STATION_ID)
recent_filename = get_station_filename(RECENT_URL, STATION_ID)

print(f"Historical file : {hist_filename}")
print(f"Recent file     : {recent_filename}")

Historical file : None
Recent file     : None


In [6]:
 # Let's see what the actual HTML looks like on the DWD page
response = requests.get(HISTORICAL_URL)

# Print first 3000 characters so we can see the structure
print(response.text[:3000])

<html>
<head><title>Index of /climate_environment/CDC/observations_germany/climate/hourly/air_temperature/historical/</title></head>
<body>
<h1>Index of /climate_environment/CDC/observations_germany/climate/hourly/air_temperature/historical/</h1><hr><pre><a href="../">../</a>
<a href="TU_Stundenwerte_Beschreibung_Stationen.txt">TU_Stundenwerte_Beschreibung_Stationen.txt</a>         27-Feb-2026 08:40:04              668559
<a href="stundenwerte_TU_00003_19500401_20110331_hist.zip">stundenwerte_TU_00003_19500401_20110331_hist.zip</a>   15-May-2025 09:45:05             2859530
<a href="stundenwerte_TU_00044_20070401_20241231_hist.zip">stundenwerte_TU_00044_20070401_20241231_hist.zip</a>   15-May-2025 09:45:18              835465
<a href="stundenwerte_TU_00052_19760101_19880101_hist.zip">stundenwerte_TU_00052_19760101_19880101_hist.zip</a>   15-May-2025 09:45:06              551483
<a href="stundenwerte_TU_00071_20091201_20191231_hist.zip">stundenwerte_TU_00071_20091201_20191231_hist.zip</

In [7]:
def get_station_filename(index_url, station_id):
    """
    Fixed version:
    - HTML is all on ONE line (no newlines), so we can't split by \n
    - Files end with _hist.zip (historical) or _akt.zip (recent)
    - We split by href=" to isolate each link
    """
    response = requests.get(index_url)
    
    # Split by href=" — gives us every link in the page
    parts = response.text.split('href="')
    
    for part in parts:
        # Grab everything before the closing quote = the filename
        link = part.split('"')[0]
        
        # Must contain our station ID AND end with .zip
        if f'_{station_id}_' in link and link.endswith('.zip'):
            print(f"✅ Found: {link}")
            return link
    
    print("❌ Not found — check station ID")
    return None

# Re-run with fixed function
hist_filename   = get_station_filename(HISTORICAL_URL, STATION_ID)
recent_filename = get_station_filename(RECENT_URL, STATION_ID)

print(f"\nHistorical file : {hist_filename}")
print(f"Recent file     : {recent_filename}")

✅ Found: stundenwerte_TU_00433_19510101_20241231_hist.zip
✅ Found: stundenwerte_TU_00433_akt.zip

Historical file : stundenwerte_TU_00433_19510101_20241231_hist.zip
Recent file     : stundenwerte_TU_00433_akt.zip


In [8]:
def download_and_extract(url, filename, save_folder):
    """
    Downloads a zip from DWD and extracts only the data CSV.
    
    Inside every DWD zip there are 3 files:
    - produkt_tu_stunde_*.txt  ← actual data (this is what we want)
    - Metadaten_*.txt          ← station metadata
    - README.txt               ← documentation
    
    We only extract the data file.
    """
    
    full_url = url + filename
    print(f"📥 Downloading: {filename}")
    
    # Download zip into memory — no need to save the zip itself
    response = requests.get(full_url, stream=True)
    
    # Check download was successful (200 = OK)
    if response.status_code != 200:
        print(f"❌ Download failed! Status code: {response.status_code}")
        return None
    
    # Wrap bytes in BytesIO so zipfile can read it like a real file
    zip_in_memory = io.BytesIO(response.content)
    
    with zipfile.ZipFile(zip_in_memory) as z:
        
        # Show all files inside the zip
        all_files = z.namelist()
        print(f"📦 Files inside zip: {all_files}")
        
        # Find the data file — always starts with 'produkt_tu_stunde_'
        data_files = [f for f in all_files if f.startswith('produkt_tu_stunde_')]
        
        if not data_files:
            print("❌ No data file found inside zip!")
            return None
        
        data_file = data_files[0]
        
        # Extract only the data file to our folder
        z.extract(data_file, save_folder)
        
        saved_path = os.path.join(save_folder, data_file)
        print(f"✅ Saved to: {saved_path}\n")
        return saved_path

# Download historical data (1951-2024) — this is ~5MB, takes ~10 seconds
hist_path = download_and_extract(
    url         = HISTORICAL_URL,
    filename    = hist_filename,
    save_folder = "data/raw/dwd_temperature/"
)

# Download recent data (last ~1.5 years)
recent_path = download_and_extract(
    url         = RECENT_URL,
    filename    = recent_filename,
    save_folder = "data/raw/dwd_temperature/"
)

📥 Downloading: stundenwerte_TU_00433_19510101_20241231_hist.zip
📦 Files inside zip: ['Metadaten_Stationsname_Betreibername_00433.html', 'Metadaten_Stationsname_Betreibername_00433.txt', 'Metadaten_Parameter_tu_stunde_00433.html', 'Metadaten_Parameter_tu_stunde_00433.txt', 'Metadaten_Geraete_Lufttemperatur_00433.html', 'Metadaten_Geraete_Lufttemperatur_00433.txt', 'Metadaten_Geraete_Rel_Feuchte_00433.html', 'Metadaten_Geraete_Rel_Feuchte_00433.txt', 'Metadaten_Geographie_00433.txt', 'Metadaten_Fehldaten_00433_19510101_20241231.html', 'Metadaten_Fehldaten_00433_19510101_20241231.txt', 'Metadaten_Fehlwerte_00433_19510101_20241231.txt', 'produkt_tu_stunde_19510101_20241231_00433.txt']
✅ Saved to: data/raw/dwd_temperature/produkt_tu_stunde_19510101_20241231_00433.txt

📥 Downloading: stundenwerte_TU_00433_akt.zip
📦 Files inside zip: ['Metadaten_Stationsname_Betreibername_00433.html', 'Metadaten_Stationsname_Betreibername_00433.txt', 'Metadaten_Parameter_tu_stunde_00433.html', 'Metadaten_Para

In [9]:
# Load both CSVs into pandas
# DWD uses semicolons as separators, not commas
df_hist   = pd.read_csv(hist_path,   sep=';', skipinitialspace=True)
df_recent = pd.read_csv(recent_path, sep=';', skipinitialspace=True)

print("=== HISTORICAL ===")
print(f"Rows    : {len(df_hist):,}")
print(f"Columns : {list(df_hist.columns)}")
print(df_hist.head(3))

print("\n=== RECENT ===")
print(f"Rows    : {len(df_recent):,}")
print(df_recent.head(3))

=== HISTORICAL ===
Rows    : 560,119
Columns : ['STATIONS_ID', 'MESS_DATUM', 'QN_9', 'TT_TU', 'RF_TU', 'eor']
   STATIONS_ID  MESS_DATUM  QN_9  TT_TU  RF_TU  eor
0          433  1951010101     5  -10.7   88.0  eor
1          433  1951010102     5  -11.0   87.0  eor
2          433  1951010103     5  -11.2   87.0  eor

=== RECENT ===
Rows    : 13,200
   STATIONS_ID  MESS_DATUM  QN_9  TT_TU  RF_TU  eor
0          433  2024082600     3   14.5   66.0  eor
1          433  2024082601     3   13.3   73.0  eor
2          433  2024082602     3   14.0   73.0  eor


In [12]:
# Stack both dataframes vertically
df_raw = pd.concat([df_hist, df_recent], ignore_index=True)

# Drop duplicates on timestamp (historical & recent overlap slightly)
df_raw = df_raw.drop_duplicates(subset=['MESS_DATUM'])

# Sort chronologically
df_raw = df_raw.sort_values('MESS_DATUM').reset_index(drop=True)

# Summary
print(f"Total rows : {len(df_raw):,}")
print(f"Date range : {df_raw['MESS_DATUM'].min()} to {df_raw['MESS_DATUM'].max()}")
print(f"Columns    : {list(df_raw.columns)}")
print(f"\nSample:\n{df_raw.head()}")

# Save combined raw file
os.makedirs("data/raw/dwd_temperature", exist_ok=True)
df_raw.to_csv("data/raw/dwd_temperature/berlin_tempelhof_raw.csv", index=False)
print("\n✅ Raw temperature data saved!")

Total rows : 570,247
Date range : 1951010101 to 2026022623
Columns    : ['STATIONS_ID', 'MESS_DATUM', 'QN_9', 'TT_TU', 'RF_TU', 'eor']

Sample:
   STATIONS_ID  MESS_DATUM  QN_9  TT_TU  RF_TU  eor
0          433  1951010101     5  -10.7   88.0  eor
1          433  1951010102     5  -11.0   87.0  eor
2          433  1951010103     5  -11.2   87.0  eor
3          433  1951010104     5  -11.4   87.0  eor
4          433  1951010105     5  -10.8   86.0  eor

✅ Raw temperature data saved!
